In [1]:
import json
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

In [7]:
data_path2 = "../data/The Hague/LOD2/Source A/Stationsbuurt_2022-04-27.json"
data_path1 = "../data/The Hague/LOD1/Source A/Stationsbuurt_2022-04-27.json"

In [8]:
with open(data_path1, "r") as f:
    cj1 = json.load(f)
with open(data_path2, "r") as f:
    cj2 = json.load(f)

In [9]:
vertices1 = np.array(cj1["vertices"], dtype=float)
if "transform" in cj1:
    scale1 = np.array(cj1["transform"]["scale"])
    translate1 = np.array(cj["transform"]["translate"])
    vertices1 = vertices1 * scale1 + translate1
vertices2 = np.array(cj2["vertices"], dtype=float)
if "transform" in cj2:
    scale2 = np.array(cj2["transform"]["scale"])
    translate2 = np.array(cj2["transform"]["translate"])
    vertices2 = vertices2 * scale2 + translate2

In [10]:
from statistics import median


def signed_area_2d(ring, _vertices):
    area = 0.0
    n = len(ring)

    for i in range(n):
        x1, y1, _ = _vertices[ring[i]]
        x2, y2, _ = _vertices[ring[(i + 1) % n]]

        area += x1 * y2 - x2 * y1

    return area / 2.0


def ensure_orientation(ring, _vertices, ccw=True):
    area = signed_area_2d(ring, _vertices)

    if ccw and area < 0:
        return ring[::-1]

    if not ccw and area > 0:
        return ring[::-1]

    return ring


def convert_to_lod1(_cj):

    _vertices = list(_cj["vertices"])

    for obj_id, city_obj in _cj.get("CityObjects", {}).items():

        new_geometries = []

        for geom in city_obj.get("geometry", []):

            if geom.get("type") != "Solid":
                new_geometries.append(geom)
                continue

            semantics = geom.get("semantics")

            if not semantics:
                new_geometries.append(geom)
                continue

            surfaces = semantics.get("surfaces", [])
            values = semantics.get("values", [])

            ground_idx = None
            roof_idx = None
            wall_idx = None

            for i, s in enumerate(surfaces):

                stype = s.get("type")

                if stype == "GroundSurface":
                    ground_idx = i

                elif stype == "RoofSurface":
                    roof_idx = i

                elif stype == "WallSurface":
                    wall_idx = i

            if ground_idx is None or roof_idx is None:
                new_geometries.append(geom)
                continue

            if wall_idx is None:
                wall_idx = len(surfaces)
                surfaces = list(surfaces)
                surfaces.append({"type": "WallSurface"})

            # --------------------------------------------------
            # Collect ground faces and roof heights
            # --------------------------------------------------

            ground_faces = []
            roof_heights = []

            for shell_i, shell in enumerate(geom["boundaries"]):

                sem_shell = values[shell_i]

                for face_i, face in enumerate(shell):

                    sem = sem_shell[face_i]

                    if sem == ground_idx:
                        ground_faces.append(face)

                    elif sem == roof_idx:

                        for ring in face:
                            for vid in ring:
                                roof_heights.append(
                                    _vertices[vid][2]
                                )

            if not ground_faces or not roof_heights:
                new_geometries.append(geom)
                continue

            lod1_height = median(roof_heights)

            # --------------------------------------------------
            # Vertex cache
            # --------------------------------------------------

            roof_vertex_cache = {}

            def get_roof_vertex(base_vid):

                if base_vid in roof_vertex_cache:
                    return roof_vertex_cache[base_vid]

                x, y, _ = _vertices[base_vid]

                roof_vid = len(_vertices)

                _vertices.append([x, y, lod1_height])

                roof_vertex_cache[base_vid] = roof_vid

                return roof_vid

            # --------------------------------------------------
            # Build shell
            # --------------------------------------------------

            shell = []
            sem_values = []

            for face in ground_faces:

                outer_ring = ensure_orientation(
                    face[0],
                    _vertices,
                    ccw=True
                )

                hole_rings = [
                    ensure_orientation(r, _vertices, ccw=False)
                    for r in face[1:]
                ]

                ground_face = [outer_ring] + hole_rings

                shell.append(ground_face)
                sem_values.append(ground_idx)

                roof_face = []

                roof_face.append(
                    [get_roof_vertex(v)
                     for v in outer_ring][::-1]
                )

                for hole in hole_rings:
                    roof_face.append(
                        [get_roof_vertex(v)
                         for v in hole][::-1]
                    )

                shell.append(roof_face)
                sem_values.append(roof_idx)

                # walls from outer boundary

                def add_walls(ring):

                    n = len(ring)

                    for i in range(n):

                        b0 = ring[i]
                        b1 = ring[(i + 1) % n]

                        t0 = get_roof_vertex(b0)
                        t1 = get_roof_vertex(b1)

                        wall = [[
                            b0,
                            b1,
                            t1,
                            t0
                        ]]

                        shell.append(wall)
                        sem_values.append(wall_idx)

                add_walls(outer_ring)

                for hole in hole_rings:
                    add_walls(hole)

            new_geom = {
                "type": "Solid",
                "lod": "1",
                "boundaries": [shell],
                "semantics": {
                    "surfaces": surfaces,
                    "values": [sem_values]
                }
            }

            new_geometries.append(new_geom)

        city_obj["geometry"] = new_geometries

    _cj["vertices"] = _vertices

    _cj.setdefault("metadata", {})
    _cj["metadata"]["datasetLod"] = "1"

    return _cj

In [11]:
import plotly.graph_objects as go
from plotly.colors import qualitative

def visualize_cityjson_3d(city_objects, vertices):
    fig = go.Figure()
    palette = qualitative.Alphabet

    for i, obj in enumerate(city_objects):
        obj_color = palette[i % len(palette)]
        obj_name = obj.get("type", f"Object_{i}")
        show_in_legend = True

        for geom in obj.get("geometry", []):
            boundaries = geom.get("boundaries", [])

            if geom["type"] == "Solid":
                boundaries = [surface for shell in boundaries for surface in shell]
            elif geom["type"] not in ["MultiSurface", "CompositeSurface"]:
                continue

            for surface in boundaries:
                outer_ring = surface[0]
                poly_pts = vertices[outer_ring]

                x = np.append(poly_pts[:, 0], poly_pts[0, 0])
                y = np.append(poly_pts[:, 1], poly_pts[0, 1])
                z = np.append(poly_pts[:, 2], poly_pts[0, 2])

                fig.add_trace(go.Scatter3d(
                    x=x, y=y, z=z,
                    mode='lines',
                    line=dict(color=obj_color, width=3),
                    name=obj_name,
                    legendgroup=str(i),
                    showlegend=show_in_legend
                ))
                show_in_legend = False

    fig.update_layout(
        title="Visualized CityObjects",
        scene=dict(aspectmode='data'),
        margin=dict(l=0, r=0, b=0, t=40),
        legend=dict(itemsizing='constant')
    )

    fig.show()

In [18]:
cj2["CityObjects"].keys()

dict_keys(['bag_0518100000229788', 'bag_0518100000286900', 'bag_0518100000259024', 'bag_0518100000348044', 'bag_0518100000206974', 'bag_0518100000241284', 'bag_0518100000328974', 'bag_0518100000322848', 'bag_0518100000306900', 'bag_0518100000330621', 'bag_0518100000337546', 'bag_0518100000332453', 'bag_0518100000316207', 'bag_0518100000334760', 'bag_0518100000250624', 'bag_0518100000284629', 'bag_0518100000213403', 'bag_0518100000278180', 'bag_0518100000331644', 'bag_0518100000328103', 'bag_0518100000261001', 'bag_0518100000208371', 'bag_0518100000319561', 'bag_0518100000214722', 'bag_0518100000204695', 'bag_0518100000314829', 'bag_0518100000290928', 'bag_0518100000337163', 'bag_0518100000338696', 'bag_0518100000207959', 'bag_0518100000297272', 'bag_0518100000249900', 'bag_0518100000319562', 'bag_0518100000278347', 'bag_0518100000271790', 'bag_0518100000213404', 'bag_0518100000305402', 'bag_0518100000259022', 'bag_0518100000246156', 'bag_0518100000282612', 'bag_0518100000228988', 'bag_

In [25]:
cto2 = [cj2["CityObjects"]['bag_0518100000241284']] # 'bag_0518100000286900' 'bag_0518100000229788' 'bag_0518100000259024', 'bag_0518100000348044', 'bag_0518100000206974', 'bag_0518100000241284'
visualize_cityjson_3d(cto2, vertices2)

In [ ]:
# cjlod1 = convert_to_lod1(cj)
# vertices_lod1 = np.array(cjlod1["vertices"], dtype=float)
# if "transform" in cjlod1:
#     scale = np.array(cjlod1["transform"]["scale"])
#     translate = np.array(cjlod1["transform"]["translate"])
#     vertices_lod1 = vertices_lod1 * scale + translate

In [26]:
cto1 = [cj1["CityObjects"]['bag_0518100000241284']]
visualize_cityjson_3d(cto1, vertices1)

In [20]:
lod1 = "../data/The Hague/temp.json"
with open(lod1, "r") as f:
    cjlod1 = json.load(f)

In [ ]:
import json
import numpy as np
import plotly.graph_objects as go

target_object_id = list(cj.get("CityObjects", {}).keys())[0]
city_obj = cj["CityObjects"][target_object_id]

fig = go.Figure()

for geom in city_obj.get("geometry", []):
    boundaries = geom.get("boundaries", [])

    if geom["type"] == "Solid":
        boundaries = [surface for shell in boundaries for surface in shell]
    elif geom["type"] not in ["MultiSurface", "CompositeSurface"]:
        continue

    for surface in boundaries:
        outer_ring = surface[0]
        poly_pts = vertices[outer_ring]

        # Appending the first vertex to the end is mathematically required 
        # to close the geometric loop of the line trace
        x = np.append(poly_pts[:, 0], poly_pts[0, 0])
        y = np.append(poly_pts[:, 1], poly_pts[0, 1])
        z = np.append(poly_pts[:, 2], poly_pts[0, 2])

        fig.add_trace(go.Scatter3d(
            x=x, y=y, z=z,
            mode='lines',
            line=dict(color='#2c3e50', width=3),
            showlegend=False
        ))

fig.update_layout(
    title=f"Rotatable Wireframe: {target_object_id}",
    scene=dict(aspectmode='data'),
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()

In [7]:
import json
import numpy as np
import torch

def create_graph_dataset(filepath):
    with open(filepath, "r") as f:
        cj = json.load(f)

    v_raw = np.array(cj["vertices"], dtype=float)
    if "transform" in cj:
        v_raw = v_raw * cj["transform"]["scale"] + cj["transform"]["translate"]

    dataset = []

    for obj_id, city_obj in cj.get("CityObjects", {}).items():
        geom_list = city_obj.get("geometry", [])
        if not geom_list:
            continue

        edges = set()
        active_vertex_indices = set()

        for geom in geom_list:
            boundaries = geom.get("boundaries", [])
            if geom["type"] == "Solid":
                boundaries = [surface for shell in boundaries for surface in shell]
            elif geom["type"] not in ["MultiSurface", "CompositeSurface"]:
                continue

            for surface in boundaries:
                ring = surface[0]
                for i in range(len(ring)):
                    active_vertex_indices.add(ring[i])
                    u = ring[i]
                    v = ring[(i + 1) % len(ring)]
                    edges.add((u, v))
                    edges.add((v, u))

        if not active_vertex_indices:
            continue

        idx_map = {old_idx: new_idx for new_idx, old_idx in enumerate(sorted(active_vertex_indices))}

        node_features = np.array([v_raw[old_idx] for old_idx in sorted(active_vertex_indices)])
        edge_index = np.array([[idx_map[u], idx_map[v]] for u, v in edges]).T

        x_tensor = torch.tensor(node_features, dtype=torch.float32)
        edge_index_tensor = torch.tensor(edge_index, dtype=torch.long)

        dataset.append({
            "id": obj_id,
            "x": x_tensor,
            "edge_index": edge_index_tensor,
            "type": city_obj.get("type", "Unknown")
        })

    return dataset

In [8]:
tensor_dataset = create_graph_dataset(data_path)

In [9]:
import json
import numpy as np
import torch

def create_sequence_dataset(filepath, sep_value=1e9):
    with open(filepath, "r") as f:
        cj = json.load(f)

    v_raw = np.array(cj["vertices"], dtype=float)
    if "transform" in cj:
        v_raw = v_raw * cj["transform"]["scale"] + cj["transform"]["translate"]

    v_raw = np.hstack((v_raw, np.zeros((v_raw.shape[0], 1))))
    sep_token = np.array([[0.0, 0.0, 0.0, 1.0]], dtype=float)

    dataset = []

    for obj_id, city_obj in cj.get("CityObjects", {}).items():
        geom_list = city_obj.get("geometry", [])
        if not geom_list:
            continue

        obj_sequence = []

        for geom in geom_list:
            boundaries = geom.get("boundaries", [])

            if geom["type"] == "Solid":
                boundaries = [surface for shell in boundaries for surface in shell]
            elif geom["type"] not in ["MultiSurface", "CompositeSurface"]:
                continue

            for surface in boundaries:
                ring = surface[0]
                ring_coords = v_raw[ring]
                obj_sequence.append(ring_coords)
                obj_sequence.append(sep_token)

        if not obj_sequence:
            continue

        seq_tensor = torch.tensor(np.vstack(obj_sequence), dtype=torch.float32)

        dataset.append({
            "id": obj_id,
            "sequence": seq_tensor,
            "type": city_obj.get("type", "Unknown")
        })

    return dataset

In [10]:
seq_dataset = create_sequence_dataset(data_path)